# validate_images_disc.ipynb

Disc指標 + MBSS指標を計算し、Best画像を選定するノートブック

**※ validate_images.ipynb との統合版**  
以前は2つのノートブックを実行する必要がありましたが、このノートブックだけで全ての指標を計算できます。

## セル構成

| # | 種類 | 内容 |
|---|------|------|
| 0 | markdown | 概要説明（このセル） |
| 1-2 | code | パラメータ設定、モデル読み込み |
| 3-4 | code | Disc指標計算関数 |
| 5 | code | MBSS計算関数 |
| 6 | code | process_one_image_disc関数（Disc + MBSS統合版） |
| 7 | code | run_inference_disc関数（MBSSスコア計算含む） |
| 8-11 | code | 単一ケースの推論 |
| 12-14 | code | バッチ推論（複数ケース） |
| 15-16 | code | Best画像選定（単一ケース） |
| 17-19 | code | **Best画像選定（バッチ） → bestimage_list_disc.xlsx** |

## 出力指標

### Disc指標
1. **disc_edge_covered**: Discの辺縁がRetina maskに完全に覆われているかどうか (True/False)
2. **disc_edge_coverage_ratio**: Discの辺縁のうちRetinaマスクに覆われている割合 (0-1)
3. **disc_area_ratio**: Disc面積のLens面積に対する比率 (%)
4. **disc_pos_ok**: Discが中心から25-75%の位置にあるか (True/False)

### MBSS指標
5. **mbss_L_multi**: マルチスケールLaplacian分散
6. **mbss_HF_ratio**: FFT高周波エネルギー比
7. **mbss_Spec_centroid**: スペクトル重心
8. **mbss_Grad_p90**: 勾配90パーセンタイル
9. **mbss_score**: MBSS統合スコア（z-score正規化後）
10. **S_mean**: 網膜領域の彩度平均

### Disc周囲シャープネス
11. **disc_core_L_multi**: Disc中心部のシャープネス
12. **disc_ring_L_multi**: Disc周辺部のシャープネス
13. **disc_core_score**: Disc中心部スコア（z-score）
14. **disc_ring_score**: Disc周辺部スコア（z-score）

## Best画像選定アルゴリズム

### 方針
1. **足切り**: `disc_edge_coverage_ratio >= 0.80` の画像のみを対象
2. **スコアリング**: `score = 0.4 × retina_ratio + 0.4 × mbss_Grad_p90 + 0.2 × mbss_score`（Min-Max正規化後）
3. **補完**: 足切りで画像数が不足する場合は `retina_ratio` のみでソートして補完

## 出力

- `validation_results_disc_{case_id}.csv`: Disc指標 + MBSS指標を含む統合CSV
- `best_images_disc_{case_id}.xlsx`: 単一ケースのBest画像リスト
- `D:\ダウンロード\bestimage_list_disc.xlsx`: 全ケースのBest画像リスト（バッチ処理時）

## 使い方

1. Cell 1-2: モデル読み込み
2. Cell 3-7: 関数定義
3. **バッチ推論**: Cell 12-14 を実行（全ケース一括処理）
4. **Best画像選定**: Cell 17-19 を実行（bestimage_list_disc.xlsx 生成）

※ 既存のCSVがある場合は推論をスキップします。再推論する場合はCSVを削除してください。

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import torch
from ultralytics import RTDETR, YOLO
from typing import Optional

# パス設定
output_root = Path(r'C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation')
validation_results_dir = output_root / 'validation_results'

# モデルパス
rtdetr_model_path = r"C:\Users\ykita\ROP_AI_project\ROP_project\models\rtdetr-l-1697_1703.pt"
yolo_seg_model_path = r"C:\Users\ykita\ROP_AI_project\ROP_project\models\yolo11n-seg_19movies.pt"

# 定数
RTDETR_CONF = 0.5
YOLO_INPUT_WIDTH = 640

# クラスID
CLS_RETINA = 0
CLS_DISC = 1
CLS_MACULA = 2

In [2]:
# モデルをロード
print("モデルを読み込み中...")
detection_model = RTDETR(rtdetr_model_path)
segmentation_model = YOLO(yolo_seg_model_path)

if torch.cuda.is_available():
    detection_model.to('cuda')
    segmentation_model.to('cuda')
    print("CUDAを使用します")
else:
    print("CPUを使用します")

print("モデル読み込み完了")

モデルを読み込み中...
CUDAを使用します
モデル読み込み完了


## Disc指標計算関数

In [3]:
def compute_disc_edge_coverage(disc_mask: np.ndarray, retina_mask: np.ndarray) -> tuple:
    """
    Discの辺縁がRetinaマスクに覆われているかを計算
    
    Returns:
    --------
    tuple: (disc_edge_covered: bool, disc_edge_coverage_ratio: float)
    """
    if disc_mask is None or retina_mask is None:
        return None, None
    
    disc_bin = (disc_mask > 0).astype(np.uint8)
    retina_bin = (retina_mask > 0).astype(np.uint8)
    
    if disc_bin.sum() == 0:
        return None, None
    
    # Discマスクの輪郭（辺縁）を抽出
    kernel = np.ones((3, 3), np.uint8)
    disc_eroded = cv2.erode(disc_bin, kernel, iterations=1)
    disc_edge = disc_bin - disc_eroded
    
    total_edge_pixels = disc_edge.sum()
    if total_edge_pixels == 0:
        return None, None
    
    # Retinaマスクを少し膨張させて境界付近でも検出
    retina_dilated = cv2.dilate(retina_bin, kernel, iterations=2)
    covered_edge_pixels = (disc_edge & retina_dilated).sum()
    
    coverage_ratio = covered_edge_pixels / total_edge_pixels
    is_covered = coverage_ratio >= 0.95
    
    return is_covered, coverage_ratio


def compute_disc_area_ratio(disc_mask: np.ndarray, lens_area: int) -> float:
    """
    Disc面積のLens面積に対する比率を計算
    
    Returns:
    --------
    float: Disc面積比率 (%)
    """
    if disc_mask is None or lens_area <= 0:
        return None
    
    disc_bin = (disc_mask > 0).astype(np.uint8)
    disc_area = disc_bin.sum()
    
    if disc_area == 0:
        return None
    
    return (disc_area / lens_area) * 100.0

In [4]:
# ==================== MBSS（画像品質特徴量）計算関数 ====================
# validate_images.ipynb から移植

from typing import Dict, Any

def to_gray_float(img_bgr_or_gray: np.ndarray) -> np.ndarray:
    """BGR/Gray いずれも float32 [0,1] グレースケールへ"""
    if img_bgr_or_gray.ndim == 3:
        gray = cv2.cvtColor(img_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_bgr_or_gray
    gray = gray.astype(np.float32)
    if gray.max() > 1.0:
        gray /= 255.0
    return gray


def laplacian_multi_var(gray01: np.ndarray, sigmas=(1.0, 2.0, 4.0), weights=(0.5, 0.3, 0.2)) -> float:
    """マルチスケール Laplacian 分散（重み付き和）"""
    vals = []
    for s, w in zip(sigmas, weights):
        ksize = int(6 * s + 1)
        if ksize % 2 == 0:
            ksize += 1
        blur = cv2.GaussianBlur(gray01, (ksize, ksize), s)
        lap = cv2.Laplacian(blur, cv2.CV_32F, ksize=3)
        vals.append(w * float(lap.var()))
    return float(np.sum(vals))


def fft_features(gray01: np.ndarray, high_freq_thresh=0.3) -> tuple:
    """FFT高周波エネルギー比とスペクトル重心"""
    h, w = gray01.shape

    wy = np.hanning(h).astype(np.float32)
    wx = np.hanning(w).astype(np.float32)
    window = np.outer(wy, wx)
    g = gray01 * window

    F = np.fft.fftshift(np.fft.fft2(g))
    mag2 = (np.abs(F) ** 2).astype(np.float64)

    cy, cx = h // 2, w // 2
    yy, xx = np.indices((h, w))
    ry = (yy - cy) / float(max(cy, 1))
    rx = (xx - cx) / float(max(cx, 1))
    r = np.sqrt(rx ** 2 + ry ** 2)
    r_norm = np.clip(r, 0, 1)

    total = mag2.sum() + 1e-12
    high_mask = r_norm > high_freq_thresh
    hf_ratio = float(mag2[high_mask].sum() / total)
    spec_centroid = float((r_norm * mag2).sum() / total)
    return hf_ratio, spec_centroid


def grad_percentile(gray01: np.ndarray, p=90) -> float:
    """勾配の90パーセンタイル"""
    gx = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    return float(np.percentile(mag, p))


def compute_mbss_components(img_bgr: np.ndarray, mask01: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Retina領域（mask）内のみで MBSS コンポーネントを算出"""
    gray = to_gray_float(img_bgr)

    mask_bool = None
    if mask01 is not None:
        if mask01.shape != gray.shape:
            mask01 = cv2.resize(mask01.astype(np.uint8), (gray.shape[1], gray.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask_bool = mask01 > 0
        if mask_bool.sum() < 100:
            return {"L_multi": None, "HF_ratio": None, "Spec_centroid": None, "Grad_p90": None, "S_mean": None}
        gray2 = gray.copy()
        gray2[~mask_bool] = 0.0
    else:
        gray2 = gray

    # 色調（HSV彩度Sの平均、網膜マスク内）
    s_mean = None
    if mask_bool is not None and img_bgr is not None and getattr(img_bgr, "ndim", 0) == 3:
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        s = hsv[:, :, 1].astype(np.float32)
        if s.max() > 1.0:
            s /= 255.0
        roi = s[mask_bool]
        if roi.size > 0:
            s_mean = float(np.mean(roi))

    return {
        "L_multi": laplacian_multi_var(gray2),
        "HF_ratio": fft_features(gray2)[0],
        "Spec_centroid": fft_features(gray2)[1],
        "Grad_p90": grad_percentile(gray2),
        "S_mean": s_mean,
    }


def compute_mbss_score(components: dict, stats: dict, weights=None) -> Optional[float]:
    """z-score正規化後、重み付き和でスコア化"""
    if any(components.get(k) is None for k in ["L_multi", "HF_ratio", "Spec_centroid", "Grad_p90"]):
        return None

    if weights is None:
        weights = {"L_multi": 0.35, "HF_ratio": 0.25, "Spec_centroid": 0.20, "Grad_p90": 0.20}

    score = 0.0
    for k, w in weights.items():
        x = float(components[k])
        m = float(stats[k]["mean"])
        s = float(stats[k]["std"]) + 1e-8
        z = (x - m) / s
        score += w * z
    return float(score)


# ==================== Disc周囲（core/ring）評価 ====================

def estimate_disc_center_radius(disc_mask01: np.ndarray):
    """discマスクから中心(cx,cy)と代表半径Rを推定"""
    m = disc_mask01.astype(np.uint8)
    if m.max() > 1:
        m = (m > 0).astype(np.uint8)

    num_labels, labels = cv2.connectedComponents(m)
    if num_labels > 1:
        areas = [(labels == i).sum() for i in range(1, num_labels)]
        main_label = int(np.argmax(areas) + 1)
        m = (labels == main_label).astype(np.uint8)

    M = cv2.moments(m)
    if M["m00"] == 0:
        return None

    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]
    area = float(m.sum())
    R = float(np.sqrt(area / np.pi))
    return cx, cy, R


def make_disc_rois(shape_hw, cx, cy, R, inner_ratio=0.6, outer_ratio=1.2):
    h, w = shape_hw
    yy, xx = np.indices((h, w))
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    core = dist < (inner_ratio * R)
    ring = (dist >= (inner_ratio * R)) & (dist < (outer_ratio * R))
    return core.astype(np.uint8), ring.astype(np.uint8)


def laplacian_multi_var_masked(gray01: np.ndarray, mask01: np.ndarray, sigmas=(1.0, 2.0, 4.0), weights=(0.5, 0.3, 0.2)) -> float:
    mask_bool = mask01.astype(bool)
    if mask_bool.sum() < 50:
        return 0.0

    vals = []
    for s, w in zip(sigmas, weights):
        ksize = int(6 * s + 1)
        if ksize % 2 == 0:
            ksize += 1
        blur = cv2.GaussianBlur(gray01, (ksize, ksize), s)
        lap = cv2.Laplacian(blur, cv2.CV_32F, ksize=3)
        roi = lap[mask_bool]
        if roi.size == 0:
            continue
        vals.append(w * float(roi.var()))
    return float(np.sum(vals)) if vals else 0.0


def compute_disc_sharpness_components(img_bgr: np.ndarray, disc_mask01: np.ndarray):
    """disc中心(core)と周辺(ring)の L_multi を返す"""
    gray = to_gray_float(img_bgr)
    est = estimate_disc_center_radius(disc_mask01)
    if est is None:
        return None, None

    cx, cy, R = est
    core_mask, ring_mask = make_disc_rois(gray.shape, cx, cy, R)

    if core_mask.sum() < 50 or ring_mask.sum() < 50:
        return None, None

    L_core = laplacian_multi_var_masked(gray, core_mask)
    L_ring = laplacian_multi_var_masked(gray, ring_mask)
    return L_core, L_ring


print("MBSS計算関数を定義しました")

MBSS計算関数を定義しました


In [5]:
def process_one_image_disc(image_path: str, detection_model, segmentation_model) -> Optional[dict]:
    """
    1枚の画像に対してDisc関連の指標 + MBSS指標を計算
    （validate_images.ipynbとvalidate_images_disc.ipynbを統合）
    """
    image = cv2.imread(image_path)
    if image is None:
        return None
    
    orig_h, orig_w = image.shape[:2]
    
    # --- Stage 1: RT-DETR Lens検出 ---
    det_results = detection_model(image, verbose=False, conf=RTDETR_CONF)
    boxes = det_results[0].boxes
    
    if len(boxes) == 0:
        return {
            'image_path': image_path,
            'lens_detected': False,
            'disc_detected': False,
            'macula_detected': False,
            'disc_edge_covered': None,
            'disc_edge_coverage_ratio': None,
            'disc_area': None,
            'disc_area_ratio': None,
            'lens_area': 0,
            'retina_area': 0,
            'retina_ratio': 0.0,
            # MBSS関連
            'mbss_L_multi': None,
            'mbss_HF_ratio': None,
            'mbss_Spec_centroid': None,
            'mbss_Grad_p90': None,
            'S_mean': None,
            # Disc周囲シャープネス
            'disc_core_L_multi': None,
            'disc_ring_L_multi': None,
            'disc_center_dist_ratio': None,
            'disc_pos_ok': None,
        }
    
    # 最大面積のboxを選択
    best_idx = 0
    best_area = 0
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        area = (x2 - x1) * (y2 - y1)
        if area > best_area:
            best_area = area
            best_idx = i
    
    x1, y1, x2, y2 = boxes[best_idx].xyxy[0].cpu().numpy().astype(int)
    
    # Lens領域をクロップ
    cropped = image[y1:y2, x1:x2]
    if cropped.size == 0:
        return {
            'image_path': image_path,
            'lens_detected': True,
            'disc_detected': False,
            'macula_detected': False,
            'disc_edge_covered': None,
            'disc_edge_coverage_ratio': None,
            'disc_area': None,
            'disc_area_ratio': None,
            'lens_area': 0,
            'retina_area': 0,
            'retina_ratio': 0.0,
            'mbss_L_multi': None,
            'mbss_HF_ratio': None,
            'mbss_Spec_centroid': None,
            'mbss_Grad_p90': None,
            'S_mean': None,
            'disc_core_L_multi': None,
            'disc_ring_L_multi': None,
            'disc_center_dist_ratio': None,
            'disc_pos_ok': None,
        }
    
    crop_h, crop_w = cropped.shape[:2]
    
    center_x = crop_w // 2
    center_y = crop_h // 2
    radius = min(center_x, center_y)
    
    # 円形マスク
    circle_mask = np.zeros((crop_h, crop_w), dtype=np.uint8)
    cv2.circle(circle_mask, (center_x, center_y), radius, 255, -1)
    lens_area = int((circle_mask > 0).sum())
    
    # マスク適用
    masked_cropped = cropped.copy()
    masked_cropped[circle_mask == 0] = 0
    
    # --- Stage 2: YOLO-seg ---
    aspect_ratio = crop_h / max(crop_w, 1)
    yolo_h = int(YOLO_INPUT_WIDTH * aspect_ratio)
    yolo_input = cv2.resize(masked_cropped, (YOLO_INPUT_WIDTH, yolo_h), interpolation=cv2.INTER_AREA)
    
    seg_results = segmentation_model(yolo_input, verbose=False, retina_masks=True)
    
    retina_mask_crop = None
    disc_mask_crop = None
    retina_area = 0
    disc_area = 0
    disc_detected = False
    macula_detected = False
    
    if seg_results and seg_results[0].masks is not None:
        r0 = seg_results[0]
        masks = r0.masks.data.cpu().numpy()
        classes = r0.boxes.cls.cpu().numpy().astype(int)
        
        for mask_data, cls_id in zip(masks, classes):
            mask_resized = cv2.resize(mask_data, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)
            mask_bin = (mask_resized > 0.5) & (circle_mask > 0)
            
            if cls_id == CLS_RETINA:
                retina_area = int(mask_bin.sum())
                retina_mask_crop = (mask_bin.astype(np.uint8) * 255)
            elif cls_id == CLS_DISC:
                disc_detected = True
                disc_area = int(mask_bin.sum())
                disc_mask_crop = (mask_bin.astype(np.uint8) * 255)
            elif cls_id == CLS_MACULA:
                macula_detected = True
    
    retina_ratio = (retina_area / lens_area * 100.0) if lens_area > 0 else 0.0
    
    # --- Disc指標計算 ---
    disc_edge_covered = None
    disc_edge_coverage_ratio = None
    disc_area_ratio = None
    
    if disc_detected and disc_mask_crop is not None:
        disc_edge_covered, disc_edge_coverage_ratio = compute_disc_edge_coverage(
            disc_mask_crop, retina_mask_crop
        )
        disc_area_ratio = compute_disc_area_ratio(disc_mask_crop, lens_area)
    
    # --- MBSS計算（Retina領域内） ---
    if retina_mask_crop is not None:
        mb = compute_mbss_components(cropped, mask01=retina_mask_crop)
    else:
        mb = {"L_multi": None, "HF_ratio": None, "Spec_centroid": None, "Grad_p90": None, "S_mean": None}
    
    # --- Disc周囲（core/ring）シャープネス ---
    disc_core_L_multi = None
    disc_ring_L_multi = None
    disc_center_dist_ratio = None
    disc_pos_ok = None
    
    if disc_mask_crop is not None:
        disc_core_L_multi, disc_ring_L_multi = compute_disc_sharpness_components(cropped, disc_mask_crop)
        est = estimate_disc_center_radius(disc_mask_crop)
        if est is not None:
            dcx, dcy, _ = est
            dist = ((dcx - center_x) ** 2 + (dcy - center_y) ** 2) ** 0.5
            disc_center_dist_ratio = float(dist / max(radius, 1))
            disc_pos_ok = (0.25 <= disc_center_dist_ratio <= 0.75)
    
    return {
        'image_path': image_path,
        'lens_detected': True,
        'disc_detected': disc_detected,
        'macula_detected': macula_detected,
        'disc_edge_covered': disc_edge_covered,
        'disc_edge_coverage_ratio': disc_edge_coverage_ratio,
        'disc_area': disc_area if disc_detected else None,
        'disc_area_ratio': disc_area_ratio,
        'lens_area': lens_area,
        'retina_area': retina_area,
        'retina_ratio': retina_ratio,
        # MBSS関連
        'mbss_L_multi': mb['L_multi'],
        'mbss_HF_ratio': mb['HF_ratio'],
        'mbss_Spec_centroid': mb['Spec_centroid'],
        'mbss_Grad_p90': mb['Grad_p90'],
        'S_mean': mb.get('S_mean'),
        # Disc周囲シャープネス
        'disc_core_L_multi': disc_core_L_multi,
        'disc_ring_L_multi': disc_ring_L_multi,
        'disc_center_dist_ratio': disc_center_dist_ratio,
        'disc_pos_ok': disc_pos_ok,
    }

In [6]:
def run_inference_disc(image_dir: str, case_id: str, detection_model, segmentation_model) -> pd.DataFrame:
    """
    指定フォルダの全画像に対して推論を実行し、MBSSスコアを計算
    """
    image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    
    results = []
    for fname in tqdm(image_files, desc=f"  {case_id} 推論中"):
        image_path = os.path.join(image_dir, fname)
        r = process_one_image_disc(image_path, detection_model, segmentation_model)
        if r is not None:
            r['image_id'] = case_id
            r['image_name'] = fname
            results.append(r)
    
    df = pd.DataFrame(results)
    
    if len(df) == 0:
        return df
    
    # -------------------- MBSSスコア算出（ケース内でz-score） --------------------
    mb_cols = ['mbss_L_multi', 'mbss_HF_ratio', 'mbss_Spec_centroid', 'mbss_Grad_p90']
    stats = {}
    for c in mb_cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(float)
        key = c.replace('mbss_', '')
        if len(vals) > 0 and float(vals.std()) > 0:
            stats[key] = {"mean": float(vals.mean()), "std": float(vals.std())}
        elif len(vals) > 0:
            stats[key] = {"mean": float(vals.mean()), "std": 1.0}
    
    # MBSS score計算
    mbss_scores = []
    for _, row in df.iterrows():
        comps = {
            "L_multi": row.get('mbss_L_multi'),
            "HF_ratio": row.get('mbss_HF_ratio'),
            "Spec_centroid": row.get('mbss_Spec_centroid'),
            "Grad_p90": row.get('mbss_Grad_p90'),
        }
        if set(stats.keys()) == {"L_multi", "HF_ratio", "Spec_centroid", "Grad_p90"}:
            mbss_scores.append(compute_mbss_score(comps, stats=stats))
        else:
            mbss_scores.append(None)
    df['mbss_score'] = mbss_scores
    
    # Disc core/ring score（z-score）
    for col_l, col_s in [('disc_core_L_multi', 'disc_core_score'), ('disc_ring_L_multi', 'disc_ring_score')]:
        if col_l not in df.columns:
            df[col_s] = None
            continue
        vals = df[col_l].dropna().astype(float)
        if len(vals) > 1 and float(vals.std()) > 0:
            m, s = float(vals.mean()), float(vals.std())
        elif len(vals) > 0:
            m, s = float(vals.mean()), 1.0
        else:
            m, s = 0.0, 1.0
        
        scores = []
        for v in df[col_l]:
            if v is None or pd.isna(v):
                scores.append(None)
            else:
                scores.append((float(v) - m) / (s + 1e-8))
        df[col_s] = scores
    
    # カラム整理
    columns_order = [
        'image_id', 'image_name', 'image_path', 'lens_detected', 'lens_area', 'retina_area', 'retina_ratio',
        'disc_detected', 'macula_detected',
        'disc_edge_covered', 'disc_edge_coverage_ratio', 'disc_area', 'disc_area_ratio',
        'mbss_L_multi', 'mbss_HF_ratio', 'mbss_Spec_centroid', 'mbss_Grad_p90', 'mbss_score',
        'S_mean',
        'disc_core_L_multi', 'disc_core_score', 'disc_ring_L_multi', 'disc_ring_score',
        'disc_center_dist_ratio', 'disc_pos_ok'
    ]
    columns_order = [c for c in columns_order if c in df.columns]
    df = df[columns_order]
    
    return df

## 単一ケースの推論

In [7]:
case_id = "1632"  # ケースIDを指定

image_dir = output_root / case_id / "images"
csv_path = validation_results_dir / f"validation_results_disc_{case_id}.csv"

print(f"case_id: {case_id}")
print(f"image_dir: {image_dir}")
print(f"csv_path: {csv_path}")

case_id: 1632
image_dir: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation\1632\images
csv_path: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation\validation_results\validation_results_disc_1632.csv


In [8]:
# CSVが存在する場合は読み込み、なければ推論実行
if csv_path.exists():
    print(f"既存のCSVを読み込み: {csv_path}")
    df = pd.read_csv(csv_path)
else:
    print(f"推論を実行します...")
    df = run_inference_disc(str(image_dir), case_id, detection_model, segmentation_model)
    df.to_csv(csv_path, index=False)
    print(f"CSVを保存しました: {csv_path}")

print(f"\n処理画像数: {len(df)}")

既存のCSVを読み込み: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation\validation_results\validation_results_disc_1632.csv

処理画像数: 676


In [9]:
# 結果の確認
print("\n=== Disc指標の統計 ===")
print(f"\nDisc検出率: {df['disc_detected'].sum()}/{len(df)} ({df['disc_detected'].mean()*100:.1f}%)")

disc_df = df[df['disc_detected'] == True].copy()
print(f"Disc検出画像数: {len(disc_df)}")

if len(disc_df) > 0:
    print(f"\ndisc_edge_covered (完全カバー): {disc_df['disc_edge_covered'].sum()}/{len(disc_df)} ({disc_df['disc_edge_covered'].mean()*100:.1f}%)")
    print(f"\ndisc_edge_coverage_ratio (カバー率):")
    print(f"  平均: {disc_df['disc_edge_coverage_ratio'].mean():.3f}")
    print(f"  中央値: {disc_df['disc_edge_coverage_ratio'].median():.3f}")
    print(f"  最小: {disc_df['disc_edge_coverage_ratio'].min():.3f}")
    print(f"  最大: {disc_df['disc_edge_coverage_ratio'].max():.3f}")
    
    print(f"\ndisc_area_ratio (Disc面積比率 %):")
    print(f"  平均: {disc_df['disc_area_ratio'].mean():.3f}")
    print(f"  中央値: {disc_df['disc_area_ratio'].median():.3f}")
    print(f"  最小: {disc_df['disc_area_ratio'].min():.3f}")
    print(f"  最大: {disc_df['disc_area_ratio'].max():.3f}")


=== Disc指標の統計 ===

Disc検出率: 183/676 (27.1%)
Disc検出画像数: 183

disc_edge_covered (完全カバー): 106/183 (60.2%)

disc_edge_coverage_ratio (カバー率):
  平均: 0.890
  中央値: 0.965
  最小: 0.000
  最大: 1.000

disc_area_ratio (Disc面積比率 %):
  平均: 3.584
  中央値: 3.173
  最小: 1.032
  最大: 8.495


## バッチ処理（複数ケース）

In [10]:
# 全ケースIDを取得
all_case_ids = []
for item in os.listdir(output_root):
    if item.isdigit() and (output_root / item / 'images').exists():
        all_case_ids.append(item)

all_case_ids = sorted(all_case_ids)
print(f"処理対象ケース: {all_case_ids}")
print(f"合計: {len(all_case_ids)} ケース")

処理対象ケース: ['1227', '1363', '1376', '1601', '1632', '1703', '1732', '1891', '1966', '2024', '2026', '2028', '2116', '2168', '2232', '2289', '2290', '2356', '2358', '2376', '2403', '2431']
合計: 22 ケース


In [11]:
# 全ケースを処理
for cid in all_case_ids:
    csv_path = validation_results_dir / f"validation_results_disc_{cid}.csv"
    image_dir = output_root / cid / "images"
    
    # 既存ファイルがある場合はスキップ
    if csv_path.exists():
        print(f"Skip {cid} (already exists)")
        continue
    
    print(f"\nProcessing {cid}...")
    df = run_inference_disc(str(image_dir), cid, detection_model, segmentation_model)
    df.to_csv(csv_path, index=False)
    print(f"保存完了: {csv_path}")

print("\n全ケース処理完了")

Skip 1227 (already exists)
Skip 1363 (already exists)
Skip 1376 (already exists)
Skip 1601 (already exists)
Skip 1632 (already exists)
Skip 1703 (already exists)
Skip 1732 (already exists)
Skip 1891 (already exists)
Skip 1966 (already exists)
Skip 2024 (already exists)
Skip 2026 (already exists)
Skip 2028 (already exists)
Skip 2116 (already exists)
Skip 2168 (already exists)
Skip 2232 (already exists)
Skip 2289 (already exists)
Skip 2290 (already exists)
Skip 2356 (already exists)
Skip 2358 (already exists)
Skip 2376 (already exists)
Skip 2403 (already exists)
Skip 2431 (already exists)

全ケース処理完了


In [12]:
# ==================== Best画像選定（単一ケース） ====================
# アルゴリズム:
# 1. disc_edge_coverage_ratio >= 0.80 で足切り
# 2. score = 0.4*retina_ratio_norm + 0.4*mbss_Grad_p90_norm + 0.2*mbss_score_norm
# 3. 足りなければ retina_ratio のみでソートして補完
#
# ※ validation_results_disc_*.csv にMBSS指標が含まれるため、
#    validation_results_*.csv とのマージは不要

# -------------------- パラメータ --------------------
case_id = "1732"  # ケースIDを指定
TOP_K_TOTAL = 30  # 最終出力の目標数

# 足切り閾値
EDGE_COVERAGE_CUTOFF = 0.80

# スコア重み
WEIGHT_RETINA_RATIO = 0.4
WEIGHT_MBSS_GRAD_P90 = 0.4
WEIGHT_MBSS_SCORE = 0.2

# -------------------- 正規化関数 --------------------
def minmax_norm(s):
    """Min-Max正規化"""
    smin, smax = s.min(), s.max()
    if smax - smin < 1e-8:
        return s * 0.0
    return (s - smin) / (smax - smin)

# -------------------- CSVファイルの読み込み --------------------
disc_csv_path = validation_results_dir / f"validation_results_disc_{case_id}.csv"

print(f"case_id: {case_id}")
print(f"disc_csv: {disc_csv_path}")

if not disc_csv_path.exists():
    raise FileNotFoundError(f"Disc CSVが見つかりません: {disc_csv_path}")

df = pd.read_csv(disc_csv_path)
print(f"\nCSV: {len(df)}件")

# -------------------- 有効データ抽出 --------------------
valid = df[
    (df['lens_detected'] == True) & 
    (df['retina_ratio'] > 0) & 
    (df['disc_detected'] == True)
].copy()

print(f"有効データ (lens+retina+disc): {len(valid)}件")

# -------------------- Stage 1: 足切り通過画像 --------------------
stage1_candidates = valid[valid['disc_edge_coverage_ratio'] >= EDGE_COVERAGE_CUTOFF].copy()

if len(stage1_candidates) > 0:
    stage1_candidates['retina_ratio_norm'] = minmax_norm(stage1_candidates['retina_ratio'].fillna(0))
    stage1_candidates['mbss_Grad_p90_norm'] = minmax_norm(stage1_candidates['mbss_Grad_p90'].fillna(0))
    stage1_candidates['mbss_score_norm'] = minmax_norm(stage1_candidates['mbss_score'].fillna(0))
    
    stage1_candidates['score'] = (
        WEIGHT_RETINA_RATIO * stage1_candidates['retina_ratio_norm'] +
        WEIGHT_MBSS_GRAD_P90 * stage1_candidates['mbss_Grad_p90_norm'] +
        WEIGHT_MBSS_SCORE * stage1_candidates['mbss_score_norm']
    )
    
    stage1_candidates = stage1_candidates.sort_values(by='score', ascending=False)
    stage1_candidates['selection_stage'] = 'Stage1_edge_cov>=0.80'
else:
    stage1_candidates = pd.DataFrame()

# -------------------- Stage 2: 足切り未通過画像 --------------------
stage2_candidates = valid[~valid.index.isin(stage1_candidates.index)].copy()

if len(stage2_candidates) > 0:
    stage2_candidates = stage2_candidates.sort_values(by='retina_ratio', ascending=False)
    stage2_candidates['selection_stage'] = 'Stage2_補完'
    stage2_candidates['score'] = None

# -------------------- 結果結合 --------------------
final_top = pd.concat([stage1_candidates, stage2_candidates], ignore_index=False)
final_top = final_top.head(TOP_K_TOTAL).reset_index(drop=True)
final_top['rank'] = range(1, len(final_top) + 1)
final_top['image_id'] = case_id

print(f"\nStage1: {len(stage1_candidates)}件, Stage2: {len(stage2_candidates)}件")
print(f"最終選定: {len(final_top)}件")

case_id: 1732
disc_csv: C:\Users\ykita\ROP_AI_project\ROP_project\bestimage_validation\validation_results\validation_results_disc_1732.csv

CSV: 137件
有効データ (lens+retina+disc): 43件

Stage1: 28件, Stage2: 15件
最終選定: 30件


In [13]:
# -------------------- 結果表示 --------------------
print("\n=== Best Top10 ===")
for i, row in final_top.head(10).iterrows():
    score_str = f"score={row['score']:.3f}" if 'score' in row and pd.notna(row.get('score')) else ""
    print(f"  {row['rank']:2d}. {row['image_name']} (retina={row['retina_ratio']:.1f}%, edge_cov={row['disc_edge_coverage_ratio']:.3f}, {score_str})")

print("\n=== Best 11-30 ===")
if len(final_top) > 10:
    for i, row in final_top.iloc[10:30].iterrows():
        print(f"  {row['rank']:2d}. {row['image_name']} (retina={row['retina_ratio']:.1f}%, edge_cov={row['disc_edge_coverage_ratio']:.3f})")
else:
    print("(なし)")

# -------------------- Excel保存 --------------------
output_best_xlsx_path = output_root / f"best_images_disc_{case_id}.xlsx"

out_cols = [
    'rank', 'image_id', 'image_name', 'selection_stage',
    'retina_ratio', 'disc_edge_coverage_ratio', 'disc_area_ratio',
    'mbss_Grad_p90', 'mbss_score', 'S_mean',
    'score'
]
out_cols = [c for c in out_cols if c in final_top.columns]

out_df = final_top[out_cols].copy()

try:
    out_df.to_excel(output_best_xlsx_path, index=False)
    print(f"\n保存しました: {output_best_xlsx_path}")
except Exception as e:
    print(f"\nExcel出力に失敗しました: {e}")
    alt_csv = output_best_xlsx_path.with_suffix('.csv')
    out_df.to_csv(alt_csv, index=False, encoding='utf-8-sig')
    print(f"代替でCSV保存しました: {alt_csv}")

out_df.head(15)


=== Best Top10 ===
   1. IMG_1732_0415.jpg (retina=93.8%, edge_cov=0.967, score=0.924)
   2. IMG_1732_0585.jpg (retina=89.9%, edge_cov=1.000, score=0.920)
   3. IMG_1732_0560.jpg (retina=89.6%, edge_cov=1.000, score=0.907)
   4. IMG_1732_0420.jpg (retina=93.7%, edge_cov=0.933, score=0.892)
   5. IMG_1732_0410.jpg (retina=91.9%, edge_cov=0.948, score=0.866)
   6. IMG_1732_0505.jpg (retina=94.4%, edge_cov=0.981, score=0.850)
   7. IMG_1732_0555.jpg (retina=88.9%, edge_cov=1.000, score=0.814)
   8. IMG_1732_0600.jpg (retina=94.0%, edge_cov=1.000, score=0.811)
   9. IMG_1732_0400.jpg (retina=92.1%, edge_cov=1.000, score=0.800)
  10. IMG_1732_0595.jpg (retina=88.8%, edge_cov=1.000, score=0.780)

=== Best 11-30 ===
  11. IMG_1732_0605.jpg (retina=93.8%, edge_cov=1.000)
  12. IMG_1732_0565.jpg (retina=89.6%, edge_cov=1.000)
  13. IMG_1732_0405.jpg (retina=90.0%, edge_cov=0.984)
  14. IMG_1732_0580.jpg (retina=89.2%, edge_cov=1.000)
  15. IMG_1732_0540.jpg (retina=90.6%, edge_cov=0.996)
  16.

,rank,image_id,image_name,selection_stage,retina_ratio,disc_edge_coverage_ratio,disc_area_ratio,mbss_Grad_p90,mbss_score,S_mean,score
0,1,1732,IMG_1732_0415.jpg,Stage1_edge_cov>=0.80,93.773543,0.966667,3.264855,0.052613,0.661723,0.716423,0.923789
1,2,1732,IMG_1732_0585.jpg,Stage1_edge_cov>=0.80,89.897282,1.000000,3.277115,0.055459,0.555524,0.654408,0.919601
2,3,1732,IMG_1732_0560.jpg,Stage1_edge_cov>=0.80,89.626188,1.000000,3.629906,0.051131,0.828005,0.748899,0.906512
3,4,1732,IMG_1732_0420.jpg,Stage1_edge_cov>=0.80,93.735369,0.933071,3.256774,0.052320,0.518097,0.734833,0.892368
4,5,1732,IMG_1732_0410.jpg,Stage1_edge_cov>=0.80,91.947002,0.948276,3.430330,0.051131,0.532301,0.724244,0.866198
5,6,1732,IMG_1732_0505.jpg,Stage1_edge_cov>=0.80,94.380730,0.980620,3.299648,0.049604,0.478603,0.728931,0.849998
6,7,1732,IMG_1732_0555.jpg,Stage1_edge_cov>=0.80,88.944686,1.000000,3.673605,0.047708,0.622047,0.789714,0.814047
7,8,1732,IMG_1732_0600.jpg,Stage1_edge_cov>=0.80,93.993068,1.000000,3.082244,0.049604,0.286018,0.763132,0.811335
8,9,1732,IMG_1732_0400.jpg,Stage1_edge_cov>=0.80,92.123744,1.000000,3.093106,0.047059,0.488789,0.735246,0.800448
9,10,1732,IMG_1732_0595.jpg,Stage1_edge_cov>=0.80,88.770034,1.000000,3.297515,0.047059,0.494087,0.703106,0.779537


---

## Best画像選定（バッチ：複数ケース）

全ケースの**全画像**を対象にスコアリングし、`bestimage_list_disc.xlsx` に出力します。

### 出力先
`D:\ダウンロード\bestimage_list_disc.xlsx`

### 出力項目（selected_images_2601113.xlsx に準拠）
`image_id`, `rank`, `image_name`, `selection_stage`, `retina_ratio`, `retina_area`, `disc_detected`, `disc_edge_coverage_ratio`, `disc_edge_covered`, `mbss_Grad_p90`, `mbss_score`, `S_mean`, `score`

In [14]:
# ==================== 全ケースの処理（全画像出力） ====================
# ※ validation_results_disc_*.csv にMBSS指標が含まれるため、
#    validation_results_*.csv とのマージは不要

# パラメータ（cell-15で定義済みだが念のため再定義）
EDGE_COVERAGE_CUTOFF = 0.80
WEIGHT_RETINA_RATIO = 0.4
WEIGHT_MBSS_GRAD_P90 = 0.4
WEIGHT_MBSS_SCORE = 0.2

def minmax_norm(s):
    """Min-Max正規化"""
    smin, smax = s.min(), s.max()
    if smax - smin < 1e-8:
        return s * 0.0
    return (s - smin) / (smax - smin)

# ケースIDリスト（cell-13で取得済み）
case_ids = all_case_ids

all_images = []

for cid in case_ids:
    print(f"\n処理中: {cid}...")
    
    # CSVファイルパス（Disc CSV = MBSS指標も含む統合版）
    disc_csv_path = validation_results_dir / f"validation_results_disc_{cid}.csv"
    
    # 存在チェック
    if not disc_csv_path.exists():
        print(f"  スキップ: Disc CSVが見つかりません")
        continue
    
    try:
        # 読み込み
        df = pd.read_csv(disc_csv_path)
        
        # 有効データ抽出（lens_detected=True かつ retina_ratio>0 かつ disc_detected=True）
        valid = df[
            (df['lens_detected'] == True) & 
            (df['retina_ratio'] > 0) & 
            (df['disc_detected'] == True)
        ].copy()
        
        if len(valid) == 0:
            print(f"  スキップ: 有効なデータがありません")
            continue
        
        # -------------------- Stage 1: 足切り通過画像 --------------------
        stage1_candidates = valid[valid['disc_edge_coverage_ratio'] >= EDGE_COVERAGE_CUTOFF].copy()
        
        if len(stage1_candidates) > 0:
            # 正規化
            stage1_candidates['retina_ratio_norm'] = minmax_norm(stage1_candidates['retina_ratio'].fillna(0))
            stage1_candidates['mbss_Grad_p90_norm'] = minmax_norm(stage1_candidates['mbss_Grad_p90'].fillna(0))
            stage1_candidates['mbss_score_norm'] = minmax_norm(stage1_candidates['mbss_score'].fillna(0))
            
            # スコア計算
            stage1_candidates['score'] = (
                WEIGHT_RETINA_RATIO * stage1_candidates['retina_ratio_norm'] +
                WEIGHT_MBSS_GRAD_P90 * stage1_candidates['mbss_Grad_p90_norm'] +
                WEIGHT_MBSS_SCORE * stage1_candidates['mbss_score_norm']
            )
            
            # ソート（スコア降順）
            stage1_candidates = stage1_candidates.sort_values(by='score', ascending=False)
            stage1_candidates['selection_stage'] = 'Stage1_edge_cov>=0.80'
        else:
            stage1_candidates = pd.DataFrame()
        
        # -------------------- Stage 2: 足切り未通過画像（全て） --------------------
        stage2_candidates = valid[~valid.index.isin(stage1_candidates.index)].copy()
        
        if len(stage2_candidates) > 0:
            # retina_ratio順にソート（降順）
            stage2_candidates = stage2_candidates.sort_values(by='retina_ratio', ascending=False)
            stage2_candidates['selection_stage'] = 'Stage2_補完'
            stage2_candidates['score'] = None  # Stage2はスコアなし
        
        # -------------------- 結果結合（全画像） --------------------
        final_all = pd.concat([stage1_candidates, stage2_candidates], ignore_index=False)
        final_all = final_all.reset_index(drop=True)
        final_all['rank'] = range(1, len(final_all) + 1)
        
        # 結果を追加（selected_images_2601113.xlsx に準拠した列順）
        for idx, row in final_all.iterrows():
            res = {
                'image_id': cid,
                'rank': row['rank'],
                'image_name': row['image_name'],
                'selection_stage': row.get('selection_stage', ''),
                'retina_ratio': row.get('retina_ratio'),
                'retina_area': row.get('retina_area'),
                'disc_detected': row.get('disc_detected'),
                'disc_edge_coverage_ratio': row.get('disc_edge_coverage_ratio'),
                'disc_edge_covered': row.get('disc_edge_covered'),
                'mbss_Grad_p90': row.get('mbss_Grad_p90'),
                'mbss_score': row.get('mbss_score'),
                'S_mean': row.get('S_mean'),
                'score': row.get('score'),
            }
            all_images.append(res)
        
        print(f"  Stage1: {len(stage1_candidates)}件, Stage2: {len(stage2_candidates)}件, 合計: {len(final_all)}件")
        
    except Exception as e:
        print(f"  エラー: {e}")
        import traceback
        traceback.print_exc()

print(f"\n処理完了: {len(case_ids)}ケース")
print(f"総画像数: {len(all_images)}件")


処理中: 1227...
  Stage1: 92件, Stage2: 13件, 合計: 105件

処理中: 1363...
  Stage1: 120件, Stage2: 7件, 合計: 127件

処理中: 1376...
  Stage1: 42件, Stage2: 58件, 合計: 100件

処理中: 1601...
  Stage1: 31件, Stage2: 9件, 合計: 40件

処理中: 1632...
  Stage1: 148件, Stage2: 28件, 合計: 176件

処理中: 1703...
  Stage1: 28件, Stage2: 36件, 合計: 64件

処理中: 1732...
  Stage1: 28件, Stage2: 15件, 合計: 43件

処理中: 1891...
  Stage1: 49件, Stage2: 7件, 合計: 56件

処理中: 1966...
  Stage1: 24件, Stage2: 5件, 合計: 29件

処理中: 2024...
  Stage1: 24件, Stage2: 42件, 合計: 66件

処理中: 2026...
  Stage1: 21件, Stage2: 4件, 合計: 25件

処理中: 2028...
  Stage1: 12件, Stage2: 28件, 合計: 40件

処理中: 2116...
  Stage1: 41件, Stage2: 5件, 合計: 46件

処理中: 2168...
  Stage1: 25件, Stage2: 24件, 合計: 49件

処理中: 2232...
  Stage1: 14件, Stage2: 14件, 合計: 28件

処理中: 2289...
  Stage1: 12件, Stage2: 10件, 合計: 22件

処理中: 2290...
  Stage1: 28件, Stage2: 0件, 合計: 28件

処理中: 2356...
  Stage1: 36件, Stage2: 57件, 合計: 93件

処理中: 2358...
  Stage1: 31件, Stage2: 20件, 合計: 51件

処理中: 2376...
  Stage1: 53件, Stage2: 10件, 合計: 63件



In [15]:
# ==================== Excel保存 ====================
# 出力先: D:\ダウンロード\bestimage_list_disc.xlsx

output_xlsx_path = Path(r"D:\ダウンロード\bestimage_list_disc.xlsx")

# DataFrameに変換
result_df = pd.DataFrame(all_images)

# 列順を整理
out_cols = [
    'image_id', 'rank', 'image_name', 'selection_stage',
    'retina_ratio', 'retina_area', 'disc_detected',
    'disc_edge_coverage_ratio', 'disc_edge_covered',
    'mbss_Grad_p90', 'mbss_score', 'S_mean', 'score'
]
out_cols = [c for c in out_cols if c in result_df.columns]
result_df = result_df[out_cols]

# Excel保存
try:
    result_df.to_excel(output_xlsx_path, index=False)
    print(f"保存しました: {output_xlsx_path}")
    print(f"総画像数: {len(result_df)}件")
except Exception as e:
    print(f"Excel保存エラー: {e}")
    # 代替でCSV保存
    alt_csv = output_xlsx_path.with_suffix('.csv')
    result_df.to_csv(alt_csv, index=False, encoding='utf-8-sig')
    print(f"代替でCSV保存しました: {alt_csv}")

# 確認表示
print("\n=== ケース別集計 ===")
print(result_df.groupby('image_id').size())

result_df.head(10)

保存しました: D:\ダウンロード\bestimage_list_disc.xlsx
総画像数: 1363件

=== ケース別集計 ===
image_id
1227    105
1363    127
1376    100
1601     40
1632    176
1703     64
1732     43
1891     56
1966     29
2024     66
2026     25
2028     40
2116     46
2168     49
2232     28
2289     22
2290     28
2356     93
2358     51
2376     63
2403     73
2431     39
dtype: int64


,image_id,rank,image_name,selection_stage,retina_ratio,retina_area,disc_detected,disc_edge_coverage_ratio,disc_edge_covered,mbss_Grad_p90,mbss_score,S_mean,score
0,1227,1,IMG_1227_1120.jpg,Stage1_edge_cov>=0.80,88.128918,314377,True,0.982684,True,0.062005,0.566334,0.655740,0.865956
1,1227,2,IMG_1227_1100.jpg,Stage1_edge_cov>=0.80,90.634129,321414,True,1.000000,True,0.057901,0.628887,0.656565,0.852234
2,1227,3,IMG_1227_1110.jpg,Stage1_edge_cov>=0.80,89.596014,313974,True,1.000000,True,0.057901,0.593148,0.659429,0.843185
3,1227,4,IMG_1227_1150.jpg,Stage1_edge_cov>=0.80,83.009568,294376,True,1.000000,True,0.056558,0.782553,0.614412,0.818190
4,1227,5,IMG_1227_1125.jpg,Stage1_edge_cov>=0.80,88.239087,314770,True,1.000000,True,0.055736,0.440168,0.672932,0.802854
5,1227,6,IMG_1227_1160.jpg,Stage1_edge_cov>=0.80,87.489011,321446,True,1.000000,True,0.052320,0.699905,0.629108,0.798449
6,1227,7,IMG_1227_1175.jpg,Stage1_edge_cov>=0.80,88.245165,322349,True,1.000000,True,0.051131,0.692936,0.627278,0.792020
7,1227,8,IMG_1227_1155.jpg,Stage1_edge_cov>=0.80,88.476589,330783,True,1.000000,True,0.049605,0.788212,0.614332,0.790803
8,1227,9,IMG_1227_1165.jpg,Stage1_edge_cov>=0.80,89.697886,335349,True,1.000000,True,0.049605,0.712808,0.630591,0.789242
9,1227,10,IMG_1227_1170.jpg,Stage1_edge_cov>=0.80,88.267390,331956,True,1.000000,True,0.050221,0.716694,0.629407,0.787263
